# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Харченко Артьом Олександрович  
**Група:** ФІТ 3-10  
**Дата:** 14.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

In [ ]:
print("Користувачів:", ratings_df["user_id"].nunique())
print("Фільмів:", ratings_df["item_id"].nunique())

ratings_df["rating"] = ratings_df["rating"].astype(float)
print("Оцінка: мін", ratings_df["rating"].min(), "| макс", ratings_df["rating"].max(), "| середня", round(ratings_df["rating"].mean(), 3))
print("Пропусків:", ratings_df.isnull().sum().sum())

print()
print("Розподіл оцінок:")
print(ratings_df["rating"].value_counts().sort_index())

**Короткі спостереження щодо даних:**

У наборі 100000 оцінок від 943 користувачів для 1682 фільмів, пропусків немає. Оцінки від 1 до 5, середня 3.53. Найчастіше ставлять 4 (34174 разів) і 3 (27145), а оцінок 1 найменше (6110), тобто люди частіше оцінюють фільми позитивно. Таблиця дуже розріджена: кожен користувач оцінив лише невелику частину фільмів (заповнено близько 6% усіх можливих пар).

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)

print("Train:", train_df.shape, "| Test:", test_df.shape)

In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)
print("Шкала оцінок:", data.reader.rating_scale)

train_data = Dataset.load_from_df(train_df[["user_id", "item_id", "rating"]], reader)
testset = list(test_df[["user_id", "item_id", "rating"]].itertuples(index=False, name=None))

print("Оцінок у train_data:", len(train_df))
print("Оцінок у testset:", len(testset))
print("Приклад з testset:", testset[0])

## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Сітки параметрів
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05],
    'random_state': [42]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD: n_factors це кількість прихованих факторів (чим більше, тим складніша модель), lr_all це швидкість навчання, reg_all це регуляризація, яка захищає від перенавчання. Для KNNBasic: k це число сусідів, min_k мінімальна кількість сусідів для прогнозу, а метрика схожості (cosine або pearson) і user_based (схожість між користувачами або між фільмами) визначають, як шукаємо схожих.

Сітки лишили невеликими, як у прикладі: для кожної моделі 8 і 16 комбінацій, щоб пошук не тривав довго. Для SVD додали random_state=42, щоб результат повторювався.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
from surprise.model_selection import KFold

# однакове розбиття для кросвалідації (3 частини, random_state=42), щоб результат повторювався
cv3 = KFold(n_splits=3, random_state=42, shuffle=True)

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=cv3, n_jobs=-1)
grid_search_svd.fit(train_data)
print("SVD: пошук завершено")

In [ ]:
grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=cv3, n_jobs=-1)
grid_search_knn.fit(train_data)
print("KNNBasic: пошук завершено")

In [ ]:
for name, gs in [("SVD", grid_search_svd), ("KNNBasic", grid_search_knn)]:
    best_idx = gs.best_index['rmse']
    print("=" * 10, name, "=" * 10)
    print("Найкращий RMSE:", round(gs.best_score['rmse'], 4))
    print("MAE при цих параметрах:", round(gs.cv_results['mean_test_mae'][best_idx], 4))
    print("Найкращий MAE (серед усіх):", round(gs.best_score['mae'], 4))
    print("Найкращі параметри за RMSE:", gs.best_params['rmse'])

**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9425 | 0.7441 | n_factors=50, lr_all=0.01, reg_all=0.05 |
| KNNBasic | 1.0272 | 0.8143 | k=40, min_k=1, cosine, user_based=True |

SVD на кросвалідації помітно краща за KNNBasic: RMSE менше приблизно на 0.085, MAE менше приблизно на 0.07. У SVD найкращими стали найбільші lr_all (0.01) і reg_all (0.05) з сітки, а в KNNBasic найбільше k (40). Тобто можливо, що з ширшою сіткою результат був би ще трохи кращим.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print("SVD параметри:", best_svd_params)
print("KNN параметри:", best_knn_params)
print("У trainset користувачів:", trainset.n_users, "| фільмів:", trainset.n_items, "| оцінок:", trainset.n_ratings)

## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print("SVD:      RMSE =", round(svd_rmse, 4), "| MAE =", round(svd_mae, 4))
print("KNNBasic: RMSE =", round(knn_rmse, 4), "| MAE =", round(knn_mae, 4))

In [ ]:
results = pd.DataFrame({
    "Model": ["SVD", "KNNBasic"],
    "RMSE": [svd_rmse, knn_rmse],
    "MAE": [svd_mae, knn_mae],
}).sort_values("RMSE").reset_index(drop=True)

display(results.round(4))

**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** на тестовій вибірці помилки трохи менші, ніж на кросвалідації (у SVD RMSE 0.919 проти 0.943, у KNNBasic 1.012 проти 1.027). Це нормально: під час кросвалідації модель вчилась на двох третинах train, а фінальну модель навчали на всьому train, тому даних було більше. Різниця невелика, і порядок моделей той самий (SVD краща), перенавчання немає.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити
user_id = '196'

# найкраща модель за CV RMSE
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model, best_name = best_svd, "SVD"
else:
    best_model, best_name = best_knn, "KNNBasic"
print("Модель для рекомендацій:", best_name)

if user_id not in trainset._raw2inner_id_users:
    print(f"Користувача {user_id} немає в навчальних даних, персональні рекомендації неможливі.")
else:
    inner_uid = trainset.to_inner_uid(user_id)
    # фільми, які користувач уже оцінив у train
    rated = {trainset.to_raw_iid(iid) for iid, _ in trainset.ur[inner_uid]}
    print("Фільмів, оцінених користувачем у train:", len(rated))

    # всі інші фільми з trainset
    all_items = [trainset.to_raw_iid(iid) for iid in trainset.all_items()]
    candidates = [iid for iid in all_items if iid not in rated]

    preds = [(iid, best_model.predict(uid=user_id, iid=iid).est) for iid in candidates]
    top10 = pd.DataFrame(preds, columns=["item_id", "predicted_rating"])
    top10 = top10.sort_values("predicted_rating", ascending=False).head(10).reset_index(drop=True)
    display(top10.round(3))

## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** За кросвалідацією обрали SVD: її CV RMSE 0.943, а в KNNBasic 1.027. Найкращі параметри для SVD: n_factors=50, lr_all=0.01, reg_all=0.05, а для KNNBasic: k=40, min_k=1, cosine, user_based=True. На тестовій вибірці SVD дала RMSE 0.919 і MAE 0.723, а KNNBasic RMSE 1.012 і MAE 0.798, тобто SVD точніша за обома метриками. Результати на тесті трохи кращі, ніж на кросвалідації (різниця близько 0.015-0.025), бо фінальну модель навчали на всьому train, і суттєвої різниці між CV і тестом немає. Помилка близько 0.9 бала за шкалою від 1 до 5 це не мало, але для простих моделей на розрідженій таблиці нормально. Обмеження рекомендацій: для нового користувача без оцінок або нового фільму модель нічого не знає і віддає приблизно середню оцінку (так званий холодний старт), а ще вона використовує тільки оцінки і не знає нічого про жанри чи опис фільмів. Також у топ-10 для користувача 196 всі прогнози близько 4.4, тобто різниця між фільмами в топі невелика.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [ ]:
# Власний рекомендаційний алгоритм: user-based колаборативна фільтрація (тільки numpy)
users = sorted(train_df["user_id"].unique(), key=int)
items = sorted(train_df["item_id"].unique(), key=int)
u_idx = {u: i for i, u in enumerate(users)}
i_idx = {m: j for j, m in enumerate(items)}

# матриця користувач x фільм (0 = не оцінено)
R = np.zeros((len(users), len(items)))
for u, m, r in train_df[["user_id", "item_id", "rating"]].itertuples(index=False, name=None):
    R[u_idx[u], i_idx[m]] = r

rated_mask = R > 0
user_mean = R.sum(axis=1) / rated_mask.sum(axis=1)
global_mean = train_df["rating"].mean()

# центруємо оцінки (віднімаємо середнє користувача) і рахуємо косинусну схожість між користувачами
R_centered = np.where(rated_mask, R - user_mean[:, None], 0)
norms = np.linalg.norm(R_centered, axis=1)
norms[norms == 0] = 1
similarity = (R_centered @ R_centered.T) / np.outer(norms, norms)
np.fill_diagonal(similarity, 0)


def my_predict(u, m, k=40):
    if u not in u_idx or m not in i_idx:
        return global_mean
    ui, mi = u_idx[u], i_idx[m]
    raters = np.where(rated_mask[:, mi])[0]
    if len(raters) == 0:
        return user_mean[ui]
    sims = similarity[ui, raters]
    top = np.argsort(-np.abs(sims))[:k]
    sims, raters = sims[top], raters[top]
    if np.abs(sims).sum() == 0:
        return user_mean[ui]
    pred = user_mean[ui] + (sims * R_centered[raters, mi]).sum() / np.abs(sims).sum()
    return float(np.clip(pred, 1, 5))


y_true = np.array([r for _, _, r in testset])
y_my = np.array([my_predict(u, m) for u, m, _ in testset])

my_rmse = np.sqrt(np.mean((y_true - y_my) ** 2))
my_mae = np.mean(np.abs(y_true - y_my))

extra = pd.DataFrame({
    "Model": ["SVD", "KNNBasic", "Власний user-based (k=40)"],
    "RMSE": [svd_rmse, knn_rmse, my_rmse],
    "MAE": [svd_mae, knn_mae, my_mae],
}).sort_values("RMSE").reset_index(drop=True)
display(extra.round(4))

**Власний алгоритм (user-based колаборативна фільтрація).** Для кожного користувача віднімаємо його середню оцінку, потім рахуємо косинусну схожість між користувачами. Оцінку фільму прогнозуємо як середню оцінку користувача плюс зважене відхилення від середнього у 40 найбільш схожих користувачів, які цей фільм оцінили. Якщо таких немає, беремо середню оцінку користувача (або загальну).

Результат на тій самій тестовій вибірці: RMSE 0.934 і MAE 0.730. Це краще за KNNBasic (1.012 і 0.798) і трохи гірше за SVD (0.919 і 0.723). Власний алгоритм виграв у KNNBasic, мабуть, тому що враховує середню оцінку кожного користувача (одні люди ставлять завжди високі оцінки, інші низькі), а KNNBasic цього не робить.